In [1]:
import json
import pandas as pd
import pprint as p
from neo4j import GraphDatabase

In [5]:
with open(r"C:\Users\Sekaran\Desktop\projects\PreJaiBot\NumpyFunctions_250.json",'r') as f:
    data = json.load(f)

In [6]:
df = pd.DataFrame()
for hier in data['hierarchy']:
    for sub_category in data['hierarchy'][hier]:
        for function in data['hierarchy'][hier][sub_category]:
            new_df = pd.DataFrame([function])
            df = pd.concat([df,new_df])

In [7]:
df.head(10)

,name,intent,category,relatedFunctions,inputs,outputs,preconditions,sideEffects,failureModes,examples
0,np.zeros,Return a new array of given shape filled with ...,"[array_creation, from_shape]","[np.ones, np.empty, np.full, np.zeros_like]","[{'name': 'shape', 'type': 'int or tuple of in...","{'type': 'np.array', 'meaning': 'Array of zero...",[shape must be a valid integer or tuple of int...,[Allocates memory for the new array],[TypeError if shape is not an integer or tuple...,"[{'description': '1D array of zeros', 'code': ..."
0,np.ones,Return a new array of given shape filled with ...,"[array_creation, from_shape]","[np.zeros, np.empty, np.full, np.ones_like]","[{'name': 'shape', 'type': 'int or tuple of in...","{'type': 'np.array', 'meaning': 'Array of ones...",[shape must be a valid integer or tuple of int...,[Allocates memory for the new array],[TypeError if shape is not an integer or tuple...,"[{'description': '1D array of ones', 'code': '..."
0,np.eye,Return a 2-D array with ones on the diagonal a...,"[array_creation, from_shape]","[np.identity, np.diag, np.zeros]","[{'name': 'N', 'type': 'int', 'meaning': 'Numb...","{'type': 'np.array', 'meaning': '2D array with...","[N must be a non-negative integer, M must be a...",[Allocates memory for the new array],"[TypeError if N or M are not integers, ValueEr...","[{'description': '3x3 identity matrix', 'code'..."
0,np.empty,Return a new array of given shape without init...,"[array_creation, from_shape]",[],"[{'name': 'x or a', 'type': 'array_like', 'mea...","{'type': 'ndarray or scalar', 'meaning': 'Resu...",[Input must be a valid NumPy array or scalar],[],[TypeError if input is of incompatible type],"[{'description': 'Basic usage', 'code': '# np...."
0,np.empty_like,Return a new array with the same shape and typ...,"[array_creation, from_shape]",[],"[{'name': 'x or a', 'type': 'array_like', 'mea...","{'type': 'ndarray or scalar', 'meaning': 'Resu...",[Input must be a valid NumPy array or scalar],[],[TypeError if input is of incompatible type],"[{'description': 'Basic usage', 'code': '# np...."
0,np.full,Return a new array of given shape filled with ...,"[array_creation, from_shape]",[],"[{'name': 'x or a', 'type': 'array_like', 'mea...","{'type': 'ndarray or scalar', 'meaning': 'Resu...",[Input must be a valid NumPy array or scalar],[],[TypeError if input is of incompatible type],"[{'description': 'Basic usage', 'code': '# np...."
0,np.full_like,Return a full array with the same shape and ty...,"[array_creation, from_shape]",[],"[{'name': 'x or a', 'type': 'array_like', 'mea...","{'type': 'ndarray or scalar', 'meaning': 'Resu...",[Input must be a valid NumPy array or scalar],[],[TypeError if input is of incompatible type],"[{'description': 'Basic usage', 'code': '# np...."
0,np.ones_like,Return an array of ones with the same shape an...,"[array_creation, from_shape]",[],"[{'name': 'x or a', 'type': 'array_like', 'mea...","{'type': 'ndarray or scalar', 'meaning': 'Resu...",[Input must be a valid NumPy array or scalar],[],[TypeError if input is of incompatible type],"[{'description': 'Basic usage', 'code': '# np...."
0,np.pad,Pad an array.,"[array_creation, from_shape]",[],"[{'name': 'x or a', 'type': 'array_like', 'mea...","{'type': 'ndarray or scalar', 'meaning': 'Resu...",[Input must be a valid NumPy array or scalar],[],[TypeError if input is of incompatible type],"[{'description': 'Basic usage', 'code': '# np...."
0,np.repeat,Repeat each element of an array.,"[array_creation, from_shape]",[],"[{'name': 'x or a', 'type': 'array_like', 'mea...","{'type': 'ndarray or scalar', 'meaning': 'Resu...",[Input must be a valid NumPy array or scalar],[],[TypeError if input is of incompatible type],"[{'description': 'Basic usage', 'code': '# np...."


In [2]:
uri = "bolt://localhost:7687"
user = "neo4j"
password = "jaikrishna24"

driver = GraphDatabase.driver(uri, auth=(user, password))

with driver.session() as session:
    result = session.run("RETURN 'Connection successful' AS msg")
    print(result.single()["msg"])

In [8]:
func_query = """
MERGE (f:Function {name: $name})
SET f.intent = $intent,
    f.preconditions = $preconditions,
    f.sideEffects = $sideEffects
"""

cat_query = """
MERGE (f:Function {name: $func})
MERGE (c:Category {name: $category})
MERGE (f)-[:BELONGS_TO]->(c)
"""

rel_query = """
MERGE (f:Function {name: $func})
MERGE (g:Function {name: $related})
MERGE (f)-[:RELATED_TO]->(g)
"""

input_query = """
MERGE (f:Function {name: $func})
MERGE (i:Input {name: $input})
MERGE (f)-[r:TAKES_INPUT]->(i)
SET r.type = $type,
    r.meaning = $meaning,
    r.constraint = $constraint
"""

output_query = """
MERGE (f:Function {name: $func})
MERGE (o:Output {type: $type})
MERGE (f)-[r:RETURNS]->(o)
SET r.meaning = $meaning
"""

error_query = """
MERGE (f:Function {name: $func})
MERGE (e:Error {name: $error})
MERGE (f)-[r:CAN_FAIL_WITH]->(e)
SET r.condition = $condition
"""


In [9]:
with driver.session() as session:

    for _, row in df.iterrows():
        session.run(func_query, {
            "name": row.get("name"),
            "intent": row.get("intent", ""),
            "preconditions": " | ".join(row.get("preconditions", [])) if row.get("preconditions") else "",
            "sideEffects": " | ".join(row.get("sideEffects", [])) if row.get("sideEffects") else ""
        })

        if row.get("category"):
            for cat in row["category"]:
                session.run(cat_query, {
                    "func": row["name"],
                    "category": cat
                })

        if row.get("relatedFunctions"):
            for rel in row["relatedFunctions"]:
                session.run(rel_query, {
                    "func": row["name"],
                    "related": rel
                })

        if row.get("inputs"):
            for inp in row["inputs"]:
                session.run(input_query, {
                    "func": row["name"],
                    "input": inp.get("name"),
                    "type": inp.get("type", ""),
                    "meaning": inp.get("meaning", ""),
                    "constraint": inp.get("constraint", "")
                })

        if row.get("outputs"):
            out = row["outputs"]
            session.run(output_query, {
                "func": row["name"],
                "type": out.get("type", ""),
                "meaning": out.get("meaning", "")
            })

        if row.get("failureModes"):
            for fail in row["failureModes"]:
                error_type = fail.split(" ")[0]
                condition = fail.replace(error_type, "").strip()

                session.run(error_query, {
                    "func": row["name"],
                    "error": error_type,
                    "condition": condition
                })
        